# CARDIA c1 xQTL

Human-operated notebook for the shared xQTL Protocol workflow. Each workflow cell submits one Snakemake controller to LSF, with its platform directives, unified `xqtl-r-base-fsusie` activation, and complete command visible in the cell. The Snakemake LSF profile submits the rule jobs; the cell output records the accepted controller job ID and the controller output goes to the named LSF logs.

Prepared longitudinal core: 1,423 people with RNA-seq and SeSAMe-complete methylation at both Y25 and Y30. Each visit has 605 Black and 769 White participants.

RNA runs through TensorQTL and SuSiE/TWAS fine-mapping. Methylation runs separately through functional SuSiE.


## Prepared inputs

Preparation job `268177996` is complete. Historical downstream jobs `268177997`–`268178000` are inactive and produced no analysis DAG or child jobs. This cell reports aggregate files without printing participant-level metadata.


In [ ]:
%%bash
cd /sc/arion/projects/load/users/sunh14/xqtl
bjobs -a -o 'jobid stat queue job_name exec_host exit_code' 268177996 268177997 268177998 268177999 268178000
tail -n 25 CARDIA/xqtl_protocol/logs/cardia_c1_core.prepare.268177996.out
printf 'VCF files: '; find CARDIA/xqtl_protocol/input/c1_eqtl/genotype -maxdepth 1 -type f -name '*.vcf.gz' | wc -l
printf 'CSI files: '; find CARDIA/xqtl_protocol/input/c1_eqtl/genotype -maxdepth 1 -type f -name '*.vcf.gz.csi' | wc -l
ls -lh CARDIA/xqtl_protocol/input/c1_eqtl/rnaseqc/*core*.gct.gz
ls -lh CARDIA/xqtl_protocol/generated/cardia_c1_eqtl.config.yaml CARDIA/xqtl_protocol/generated/cardia_c1_mqtl.config.yaml


Expected: 22 VCFs, 22 CSI files, four count GCTs, four TPM GCTs, and both configs. The preparation log reports 60,746 genes and 760,181 methylation probes.

## Shared CARDIA C1 genotype location

All analysis graphs in this notebook use one neutral C1 genotype directory. The setup cell preserves the existing genotype work, then points both the RNA and methylation graph paths to it. Run genotype-producing analysis graphs one at a time so only one controller writes this shared location.


In [ ]:
%%bash
set -euo pipefail
shared_genotype=/sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/work/c1_shared/data_preprocessing/genotype
eqtl_genotype=/sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/work/c1_eqtl/data_preprocessing/genotype
mqtl_genotype=/sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/work/c1_mqtl/data_preprocessing/genotype
mkdir -p "$(dirname "$shared_genotype")" "$(dirname "$eqtl_genotype")" "$(dirname "$mqtl_genotype")"
if [[ ! -e "$shared_genotype" ]]; then
  if [[ -d "$eqtl_genotype" && ! -L "$eqtl_genotype" ]]; then
    mv "$eqtl_genotype" "$shared_genotype"
  else
    mkdir -p "$shared_genotype"
  fi
fi
ln -sfn "$shared_genotype" "$eqtl_genotype"
ln -sfn "$shared_genotype" "$mqtl_genotype"
readlink -f "$eqtl_genotype"
readlink -f "$mqtl_genotype"


## RNA eQTL dry-run

This cell submits the exact shared-Snakefile dry-run to LSF. The target appears before `--set-resources` because that option accepts multiple values.


In [ ]:
%%bash
mkdir -p /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs
bsub <<'LSF'
#BSUB -J cardia_c1_eqtl_xqtl_core_dryrun
#BSUB -P acc_LOAD
#BSUB -q premium
#BSUB -W 04:00
#BSUB -n 1
#BSUB -R "span[hosts=1] rusage[mem=40GB]"
#BSUB -oo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_eqtl.xqtl_core.dryrun.%J.out
#BSUB -eo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_eqtl.xqtl_core.dryrun.%J.err
set -eo pipefail
source /sc/arion/work/sunh14/conda/mambaforge/etc/profile.d/conda.sh
conda activate /sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie
export PATH="/sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie/bin:${PATH}"
hash -r
python -m snakemake \
  --snakefile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/Snakefile \
  --configfile /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/generated/cardia_c1_eqtl.config.yaml \
  --directory /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/work/c1_eqtl \
  --profile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/profiles/lsf \
  --cores 72 --jobs 100 xqtl_core \
  --set-resources vcf_qc:lsf_queue=express vcf_to_plink:lsf_queue=express \
  merge_plink:lsf_queue=express plink_qc:lsf_queue=express \
  genotype_by_chrom:lsf_queue=express king_kinship:lsf_queue=express \
  tensorqtl_cis:lsf_queue=express \
  --dry-run
LSF


## RNA eQTL production and resume

Run this cell to start the analysis. The same cell is the resume command: rerun it without force after a failure or interrupted controller, and Snakemake reuses valid outputs. The endpoint is `xqtl_core`, including SuSiE/TWAS fine-mapping and excluding optional plots.


In [ ]:
%%bash
mkdir -p /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs
bsub <<'LSF'
#BSUB -J cardia_c1_eqtl_xqtl_core
#BSUB -P acc_LOAD
#BSUB -q premium
#BSUB -W 144:00
#BSUB -n 1
#BSUB -R "span[hosts=1] rusage[mem=40GB]"
#BSUB -oo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_eqtl.xqtl_core.%J.out
#BSUB -eo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_eqtl.xqtl_core.%J.err
set -eo pipefail
source /sc/arion/work/sunh14/conda/mambaforge/etc/profile.d/conda.sh
conda activate /sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie
export PATH="/sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie/bin:${PATH}"
hash -r
python -m snakemake \
  --snakefile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/Snakefile \
  --configfile /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/generated/cardia_c1_eqtl.config.yaml \
  --directory /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/work/c1_eqtl \
  --profile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/profiles/lsf \
  --cores 72 --jobs 100 xqtl_core \
  --set-resources vcf_qc:lsf_queue=express vcf_to_plink:lsf_queue=express \
  merge_plink:lsf_queue=express plink_qc:lsf_queue=express \
  genotype_by_chrom:lsf_queue=express king_kinship:lsf_queue=express \
  tensorqtl_cis:lsf_queue=express
LSF


The cell returns after `bsub` accepts the controller; the controller LSF job remains active while Snakemake manages its LSF rules. For a failure, use the first failing rule and its native log, fix the smallest config or shared-contract defect, and rerun the production cell. Do not delete outputs or add another workflow.

## RNA completion

A successful controller and requested outputs are separate facts. The four RNA themes must each have TensorQTL and SuSiE/TWAS outputs.


In [ ]:
%%bash
cd /sc/arion/projects/load/users/sunh14/xqtl
find CARDIA/xqtl_protocol/work/c1_eqtl/association_scan -path '*/cardia_pbmc_*_c1_core_*/TensorQTL/.done_cis' -type f -print | sort
find CARDIA/xqtl_protocol/work/c1_eqtl/finemapping -path '*/cardia_pbmc_*_c1_core_*/susie_twas/.done_susie_twas' -type f -print | sort
find CARDIA/xqtl_protocol/work/c1_eqtl/finemapping -path '*/cardia_pbmc_*_c1_core_*/susie_twas/susie_twas.outputs.tsv' -type f -size +0c -print | sort


Expected: four `.done_cis`, four `.done_susie_twas`, and four nonempty `susie_twas.outputs.tsv` files.

## Methylation fSuSiE dry-run

Run methylation as a separate output graph after the genotype-reuse cell. The endpoint is functional SuSiE (`methylation_fsusie`); TensorQTL is not part of this graph.


In [ ]:
%%bash
mkdir -p /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs
bsub <<'LSF'
#BSUB -J cardia_c1_mqtl_fsusie_dryrun
#BSUB -P acc_LOAD
#BSUB -q premium
#BSUB -W 04:00
#BSUB -n 1
#BSUB -R "span[hosts=1] rusage[mem=40GB]"
#BSUB -oo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_mqtl.fsusie.dryrun.%J.out
#BSUB -eo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_mqtl.fsusie.dryrun.%J.err
set -eo pipefail
source /sc/arion/work/sunh14/conda/mambaforge/etc/profile.d/conda.sh
conda activate /sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie
export PATH="/sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie/bin:${PATH}"
hash -r
python -m snakemake \
  --snakefile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/Snakefile \
  --configfile /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/generated/cardia_c1_mqtl.config.yaml \
  --directory /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/work/c1_mqtl \
  --profile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/profiles/lsf \
  --cores 72 --jobs 100 methylation_fsusie \
  --set-resources vcf_qc:lsf_queue=express vcf_to_plink:lsf_queue=express \
  merge_plink:lsf_queue=express plink_qc:lsf_queue=express \
  genotype_by_chrom:lsf_queue=express king_kinship:lsf_queue=express \
  methylation_annotate:lsf_queue=express methylation_soft_impute:lsf_queue=express \
  methylation_fsusie_manifest:lsf_queue=express methylation_qtl_dataset:lsf_queue=express \
  methylation_fsusie_fit:lsf_queue=express \
  --dry-run
LSF


## Methylation fSuSiE production and resume


In [ ]:
%%bash
mkdir -p /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs
bsub <<'LSF'
#BSUB -J cardia_c1_mqtl_fsusie
#BSUB -P acc_LOAD
#BSUB -q premium
#BSUB -W 144:00
#BSUB -n 1
#BSUB -R "span[hosts=1] rusage[mem=40GB]"
#BSUB -oo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_mqtl.fsusie.%J.out
#BSUB -eo /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/logs/cardia_c1_mqtl.fsusie.%J.err
set -eo pipefail
source /sc/arion/work/sunh14/conda/mambaforge/etc/profile.d/conda.sh
conda activate /sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie
export PATH="/sc/arion/work/sunh14/conda/envs/xqtl-r-base-fsusie/bin:${PATH}"
hash -r
python -m snakemake \
  --snakefile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/Snakefile \
  --configfile /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/generated/cardia_c1_mqtl.config.yaml \
  --directory /sc/arion/projects/load/users/sunh14/xqtl/CARDIA/xqtl_protocol/work/c1_mqtl \
  --profile /sc/arion/projects/load/users/sunh14/xqtl/xqtl-protocol/code/snakemake/profiles/lsf \
  --cores 72 --jobs 100 methylation_fsusie \
  --set-resources vcf_qc:lsf_queue=express vcf_to_plink:lsf_queue=express \
  merge_plink:lsf_queue=express plink_qc:lsf_queue=express \
  genotype_by_chrom:lsf_queue=express king_kinship:lsf_queue=express \
  methylation_annotate:lsf_queue=express methylation_soft_impute:lsf_queue=express \
  methylation_fsusie_manifest:lsf_queue=express methylation_qtl_dataset:lsf_queue=express \
  methylation_fsusie_fit:lsf_queue=express
LSF


mQTL completion requires four native `<theme>/methylation_fsusie/.done_fsusie` endpoints. SeSAMe annotation and soft imputation remain shared Snakemake rules; TensorQTL is not part of this graph.
